In [1]:
import pandas as pd
import geopandas as gpd
from typing import Final
from blocksnet.enums import LandUse


In [2]:
basline_blocks = gpd.read_file('../data/gatchina/blocks_clean_gatchina.geojson')

target_id = 86
target_block = basline_blocks.loc[basline_blocks["id"] == target_id]
basline_blocks["is_project"] = False
basline_blocks.loc[basline_blocks["id"] == target_id, "is_project"] = True


In [3]:
basline_blocks.loc[basline_blocks["id"] == target_id].iloc[0]

residential                                                      0.268468
business                                                         0.648297
recreation                                                       0.065871
industrial                                                            0.0
transport                                                         0.01739
special                                                               0.0
agriculture                                                           0.0
land_use                                                 LandUse.BUSINESS
share                                                            0.648297
footprint_area                                               47096.975151
build_floor_area                                            107107.690178
living_area                                                  44241.785252
non_living_area                                              62865.904925
population                            

In [4]:

from blocksnet.enums import LandUse

params_repaired = {'footprint_area': 11837.651977125355,
 'l': 5.8767055962839265,
 'mxi': 0.1375379812412612,
 'residential': 0.5248496543731239,
 'business': 0.07003065068175769,
 'recreation': 0.13436666513537474,
 'industrial': 0.09660322396689847,
 'transport': 0.10346542264567449,
 'special': 0.0703581018232481,
 'agriculture': 0.0003262813739226394,
 'share': 0.5248496543731239,
 'land_use': LandUse.RESIDENTIAL,
 'build_floor_area': 69566.39562083407,
 'living_area': 9568.021615920432,
 'non_living_area': 59998.37400491364,
 'population': 478.4010807960216,
 'fsi': 0.2652664851041709,
 'gsi': 0.045138637755124125,
 'morphotype': 'mid-rise non-residential'}

In [5]:
from urbanomy.methods.land_value_modeling import (
    ScenarioTEPModifier,
)
import pandas as pd


modifier = ScenarioTEPModifier(basline_blocks)
blocks_gen = modifier.apply(target_id, params_repaired)

gen_blocks = pd.DataFrame(blocks_gen)
gen_blocks.loc[gen_blocks["id"] == target_id].iloc[0]

residential                                                          0.52485
business                                                            0.070031
recreation                                                          0.134367
industrial                                                          0.096603
transport                                                           0.103465
special                                                             0.070358
agriculture                                                         0.000326
land_use                                                 LandUse.RESIDENTIAL
share                                                                0.52485
footprint_area                                                  11837.651977
build_floor_area                                                69566.395621
living_area                                                      9568.021616
non_living_area                                                 59998.374005

In [6]:
from urbanomy.methods.investment_potential import prepare_investment_input

investment_input = prepare_investment_input(
    gdf = basline_blocks
)

investment_input.head()


2026-03-13 19:17:54.266 | WARNING  | urbanomy.utils.investment_input:prepare_investment_input:280 - prepare_investment_input: колонка 'land_value_before' не найдена; значение оставлено пустым, текущая цена записана в 'land_value_after'.


,land_use,land_value,land_value_after,land_value_before,residential,business,recreation,industrial,transport,special,agriculture,site_area,living_area,non_living_area,build_floor_area,land_use_before,build_floor_area_before
0,LandUse.BUSINESS,6.519223e+08,6.519223e+08,NaN,70405.906896,170016.485982,17274.763974,0.0,4560.664101,0.0,0.0,262250.979778,44241.785252,62865.904925,107107.690178,LandUse.BUSINESS,107107.690178


## Инвевстиционная привлекательность

In [7]:
from blocksnet.enums import LandUse

benchmarks_demo = {
    LandUse.RESIDENTIAL: {
        "cost_build": 45_000,
        "price_sale": 140_000,
        "construction_years": 3,
        "sale_years": 4,
        "opex_rate": 800,
        "cost_demolition": 900,  # ₽/м² ориентировочно по России
    },
    LandUse.BUSINESS: {
        "cost_build": 55_000,
        "rent_annual": 25_000,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_300,
        "cost_demolition": 900,
    },
    LandUse.RECREATION: {
        "cost_build": 20_000,
        "rent_annual": 7_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_000,
        "cost_demolition": 900,
    },
    LandUse.SPECIAL: {
        "cost_build": 35_000,
        "rent_annual": 11_000,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 1_500,
        "cost_demolition": 900,
    },
    LandUse.INDUSTRIAL: {
        "cost_build": 38_000,
        "rent_annual": 14_800,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 700,
        "cost_demolition": 900,
    },
    LandUse.AGRICULTURE: {
        "cost_build": 25_000,
        "rent_annual": 6_500,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 300,
        "cost_demolition": 900,
    },
    LandUse.TRANSPORT: {
        "cost_build": 18_000,
        "rent_annual": 8_200,
        "rent_years": 15,
        "construction_years": 3,
        "opex_rate": 600,
        "cost_demolition": 900,
    },
}



In [8]:
from urbanomy.methods.investment_potential import InvestmentAttractivenessAnalyzer

an = InvestmentAttractivenessAnalyzer(benchmarks=benchmarks_demo)
summary = an.calculate_investment_metrics(investment_input, discount_rate=0.18)
summary


Project totals:
 • Land area:          262,257.82
 • Built area:         107,107.69
 • Land value:         651,922,266.23
 • Demolition cost:    96,396,921.16
 • Construction cost:  5,195,090,598.05
 • Investment need:    5,943,409,785.43
 • Project NPV:        2,832,533,920.13
 • Project IRR:        0.29
 • Project PI:         1.56
 • Project PP (yrs):   6.56


,land_use,land_area,land_value_before,built_area,land_value,demolition_cost,construction_cost,investment_need,NPV,IRR,PI,PP_years,EI
0,LandUse.BUSINESS,262257.82,NaN,107107.69,6.519223e+08,96396921.16,5.195091e+09,5.943410e+09,2.832534e+09,0.29,1.56,6.56,96.08


In [9]:
# scn.to_file('../data/blocks_investment.geojson', driver='GeoJSON')